# Step 1: Import and read CSV

In [1]:

import pandas as pd
import numpy as np

file_path = "I_B1_2a_36_Mean_Science_Performance.xlsx"

# Read the raw Excel file without assigning column headers
raw_df = pd.read_excel(file_path, sheet_name="Sheet1", header=None)

print("Original shape:", raw_df.shape)
display(raw_df.head(15))

Original shape: (112, 30)


,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
0,Table I.B1.2a.36,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Mean science performance, 2006 through 2025",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,"Science performance, by PISA cycle",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Average decennial trend in science performance¹,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,since 2015,NaN,NaN
8,NaN,PISA 2006,NaN,PISA 2009,NaN,PISA 2012,NaN,PISA 2015,NaN,PISA 2018,...,NaN,PISA 2015,NaN,PISA 2018,NaN,PISA 2022,NaN,(or earliest assessment available after 2015),NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,(PISA 2025 - PISA 2015),NaN,(PISA 2025 - PISA 2018),NaN,(PISA 2025 - PISA 2022),NaN,NaN,NaN,NaN


# Step 2: Choose only necessary columns 

In [2]:

# Column positions identified from the original Excel file
year_columns = {
    0: "Country",
    1: "2006",
    3: "2009",
    5: "2012",
    7: "2015",
    9: "2018",
    11: "2022",
    13: "2025"
}

# Select relevant columns and actual data rows
science_df = raw_df.iloc[11:104, list(year_columns.keys())].copy()

# Rename columns
science_df.columns = list(year_columns.values())

science_df = science_df.reset_index(drop=True)

display(science_df.head(10))


,Country,2006,2009,2012,2015,2018,2022,2025
0,OECD average,m,m,496,489,m,m,482
1,OECD average-23,503,506,505,497,493,491,486
2,OECD average-35,495,m,499,491,489,487,484
3,Argentina,391,401,406,m,404,406,393
4,Armenia,m,m,m,m,m,m,369
5,Australia,527,527,521,510,503,507,509
6,Austria,511,m,506,495,490,491,497
7,Azerbaijan,382,373,m,m,m,m,409
8,Belgium,510,507,505,502,499,491,490
9,Brazil,390,405,402,401,404,403,409


# Step 3: Handle missing values and different types of variables

In [3]:

# Remove extra spaces in country names
science_df["Country"] = science_df["Country"].astype(str).str.strip()

# Remove footnote markers from country names
science_df["Country"] = science_df["Country"].str.replace(
    r"\*+$", "", regex=True
).str.strip()

# Identify score columns
score_cols = [str(y) for y in [2006, 2009, 2012, 2015, 2018, 2022, 2025]]

# Convert non-numeric values (including 'm') to NaN
for col in score_cols:
    science_df[col] = pd.to_numeric(
        science_df[col], errors="coerce"
    )

# Check missing values
print("Missing values:")
print(science_df[score_cols].isna().sum())

print("\nColumn types:")
print(science_df.dtypes)

display(science_df.head())


Missing values:
2006    38
2009    30
2012    32
2015    29
2018    17
2022    15
2025     0
dtype: int64

Column types:
Country        str
2006       float64
2009       float64
2012       float64
2015       float64
2018       float64
2022       float64
2025         int64
dtype: object


,Country,2006,2009,2012,2015,2018,2022,2025
0,OECD average,NaN,NaN,496.0,489.0,NaN,NaN,482
1,OECD average-23,503.0,506.0,505.0,497.0,493.0,491.0,486
2,OECD average-35,495.0,NaN,499.0,491.0,489.0,487.0,484
3,Argentina,391.0,401.0,406.0,NaN,404.0,406.0,393
4,Armenia,NaN,NaN,NaN,NaN,NaN,NaN,369


# Step 4: Separate OECD aveages from countries data

In [4]:

# Keep OECD averages separately
oecd_df = science_df[
    science_df["Country"].str.startswith("OECD average")
].copy()

# Keep countries and participating education systems
country_df = science_df[
    ~science_df["Country"].str.startswith("OECD average")
].copy()

# Remove rows where all science scores are missing
country_df = country_df.dropna(
    subset=score_cols,
    how="all"
).reset_index(drop=True)

print("Country/system rows:", len(country_df))
print("OECD average rows:", len(oecd_df))

display(country_df.head())


Country/system rows: 90
OECD average rows: 3


,Country,2006,2009,2012,2015,2018,2022,2025
0,Argentina,391.0,401.0,406.0,NaN,404.0,406.0,393
1,Armenia,NaN,NaN,NaN,NaN,NaN,NaN,369
2,Australia,527.0,527.0,521.0,510.0,503.0,507.0,509
3,Austria,511.0,NaN,506.0,495.0,490.0,491.0,497
4,Azerbaijan,382.0,373.0,NaN,NaN,NaN,NaN,409


# Step 5: Create Science Score Change variable

In [5]:

# Calculate changes in mean science performance
country_df["Science_Change_2022_2025"] = (
    country_df["2025"] - country_df["2022"]
)

# Calculate long-term change
country_df["Science_Change_2006_2025"] = (
    country_df["2025"] - country_df["2006"]
)

# Check results
display(
    country_df[
        ["Country", "2022", "2025", "Science_Change_2022_2025"]
    ].head(15)
)

# Verify Australian values
display(country_df[country_df["Country"] == "Australia"])


,Country,2022,2025,Science_Change_2022_2025
0,Argentina,406.0,393,-13.0
1,Armenia,NaN,369,NaN
2,Australia,507.0,509,2.0
3,Austria,491.0,497,6.0
4,Azerbaijan,NaN,409,NaN
5,Belgium,491.0,490,-1.0
6,Brazil,403.0,409,6.0
7,Brunei Darussalam,446.0,439,-7.0
8,B-S-J-Z (China),NaN,597,NaN
9,Bulgaria,421.0,421,0.0


,Country,2006,2009,2012,2015,2018,2022,2025,Science_Change_2022_2025,Science_Change_2006_2025
2,Australia,527.0,527.0,521.0,510.0,503.0,507.0,509,2.0,-18.0


# Step 6: Convert science data into long format. The purpose of this step is that: science_long will be able to plot line chart over time, and country_df is essential when merging and calculating science score changes

In [6]:

# Convert from wide to long format
science_long = country_df.melt(
    id_vars="Country",
    value_vars=score_cols,
    var_name="Year",
    value_name="Science_Score"
)

# Remove rows with missing science scores
science_long = science_long.dropna(
    subset=["Science_Score"]
).copy()

# Convert Year to integer
science_long["Year"] = science_long["Year"].astype(int)

# Sort data by country and year
science_long = science_long.sort_values(
    by=["Country", "Year"]
).reset_index(drop=True)

print("Long format shape:", science_long.shape)
display(science_long.head(15))


Long format shape: (474, 3)


,Country,Year,Science_Score
0,Argentina,2006,391.0
1,Argentina,2009,401.0
2,Argentina,2012,406.0
3,Argentina,2018,404.0
4,Argentina,2022,406.0
5,Argentina,2025,393.0
6,Armenia,2025,369.0
7,Australia,2006,527.0
8,Australia,2009,527.0
9,Australia,2012,521.0


# Step 7: Export cleaned science data

In [7]:

# Check duplicate countries
print("Duplicated countries:",
      country_df["Country"].duplicated().sum())

# Export datasets
country_df.to_csv("science_wide.csv", index=False)
science_long.to_csv("science_long.csv", index=False)
oecd_df.to_csv("science_oecd_averages.csv", index=False)

print("Science datasets exported successfully!")


Duplicated countries: 0
Science datasets exported successfully!


# Step 8: Clean Digital Resource Shortage

In [8]:
digital_raw = pd.read_excel(
    "I_B1_4_9_Digital_Resource_shortage.xlsx",
    sheet_name="Sheet1",
    header=None
)

print("Original shape:", digital_raw.shape)

# Data rows: Excel rows 10–102
# Includes OECD average and country/system observations
digital_df = digital_raw.iloc[9:102, [0, 1, 4, 7, 10]].copy()

digital_df.columns = [
    "Country",
    "Digital_Lack_2022",
    "Digital_Quality_2022",
    "Digital_Lack_2025",
    "Digital_Quality_2025"
]

# Standardise country names
digital_df["Country"] = (
    digital_df["Country"]
    .astype("string")
    .str.strip()
    .str.replace(r"\*+$", "", regex=True)
    .str.strip()
)

digital_cols = [
    "Digital_Lack_2022",
    "Digital_Quality_2022",
    "Digital_Lack_2025",
    "Digital_Quality_2025"
]

# Convert missing and non-numeric values to NaN
for col in digital_cols:
    digital_df[col] = pd.to_numeric(
        digital_df[col],
        errors="coerce"
    )

# Remove empty and invalid records
digital_df = digital_df.dropna(subset=["Country"])
digital_df = digital_df.dropna(
    subset=digital_cols,
    how="all"
)

# Separate OECD aggregate
digital_oecd = digital_df[
    digital_df["Country"].str.startswith("OECD average")
].copy()

digital_df = digital_df[
    ~digital_df["Country"].str.startswith("OECD average")
].copy()

# Calculate changes (percentage points)
digital_df["Digital_Lack_Change"] = (
    digital_df["Digital_Lack_2025"] -
    digital_df["Digital_Lack_2022"]
)

digital_df["Digital_Quality_Change"] = (
    digital_df["Digital_Quality_2025"] -
    digital_df["Digital_Quality_2022"]
)

digital_df = digital_df.reset_index(drop=True)

# Validation
assert not digital_df["Country"].duplicated().any()

for col in digital_cols:
    assert digital_df[col].dropna().between(0, 100).all()

print("Cleaned shape:", digital_df.shape)
print("\nMissing values:")
print(digital_df.isna().sum())

display(digital_df.head(10))
display(digital_df[digital_df["Country"] == "Australia"])

# Export
digital_df.to_csv("digital_clean.csv", index=False)
digital_oecd.to_csv("digital_oecd.csv", index=False)

print("Step 8 completed!")

Original shape: (106, 19)
Cleaned shape: (90, 7)

Missing values:
Country                    0
Digital_Lack_2022         12
Digital_Quality_2022      12
Digital_Lack_2025          0
Digital_Quality_2025       0
Digital_Lack_Change       12
Digital_Quality_Change    12
dtype: int64


,Country,Digital_Lack_2022,Digital_Quality_2022,Digital_Lack_2025,Digital_Quality_2025,Digital_Lack_Change,Digital_Quality_Change
0,Argentina,67.72479,67.53960,62.33416,60.49732,-5.39063,-7.04228
1,Armenia,NaN,NaN,55.76785,53.62173,NaN,NaN
2,Australia,9.92705,9.85572,11.75857,11.36550,1.83152,1.50978
3,Austria,25.90449,20.69598,19.22027,19.59430,-6.68422,-1.10168
4,Azerbaijan,NaN,NaN,47.73195,44.48527,NaN,NaN
5,Belgium,17.04771,19.73607,10.77647,12.12294,-6.27124,-7.61313
6,Brazil,34.48099,34.53144,35.73083,37.33292,1.24984,2.80148
7,Brunei Darussalam,50.63843,49.59211,44.16915,46.01484,-6.46928,-3.57727
8,B-S-J-Z (China),NaN,NaN,32.06114,32.11765,NaN,NaN
9,Bulgaria,9.35496,8.05109,11.66984,12.64407,2.31488,4.59298


,Country,Digital_Lack_2022,Digital_Quality_2022,Digital_Lack_2025,Digital_Quality_2025,Digital_Lack_Change,Digital_Quality_Change
2,Australia,9.92705,9.85572,11.75857,11.3655,1.83152,1.50978


Step 8 completed!


# Step 9: Socioeconomic Inequality Datacleaning

In [9]:
inequality_raw = pd.read_excel(
    "I_B1_4_7_Socioeconomic_inequality.xlsx",
    sheet_name="Sheet1",
    header=None
)

print("Original shape:", inequality_raw.shape)

# Select 2025 socioeconomic-profile data
# 0 = Country
# 1 = Overall average
# 4 = Disadvantaged schools
# 7 = Socioeconomically average schools
# 10 = Advantaged schools

inequality_df = inequality_raw.iloc[
    9:102, [0, 1, 4, 7, 10]
].copy()

inequality_df.columns = [
    "Country",
    "Overall_Digital_Lack_2025",
    "Disadvantaged_2025",
    "Average_Schools_2025",
    "Advantaged_2025"
]

# Standardise country names
inequality_df["Country"] = (
    inequality_df["Country"]
    .astype("string")
    .str.strip()
    .str.replace(r"\*+$", "", regex=True)
    .str.strip()
)

inequality_cols = [
    "Overall_Digital_Lack_2025",
    "Disadvantaged_2025",
    "Average_Schools_2025",
    "Advantaged_2025"
]

# Convert numeric columns
for col in inequality_cols:
    inequality_df[col] = pd.to_numeric(
        inequality_df[col],
        errors="coerce"
    )

# Remove empty and invalid records
inequality_df = inequality_df.dropna(subset=["Country"])
inequality_df = inequality_df.dropna(
    subset=inequality_cols,
    how="all"
)

# Separate OECD aggregate
inequality_oecd = inequality_df[
    inequality_df["Country"].str.startswith("OECD average")
].copy()

inequality_df = inequality_df[
    ~inequality_df["Country"].str.startswith("OECD average")
].copy()

# Calculate socioeconomic gap
# Positive = disadvantaged schools report more shortages
inequality_df["Socioeconomic_Gap_2025"] = (
    inequality_df["Disadvantaged_2025"] -
    inequality_df["Advantaged_2025"]
)

inequality_df = inequality_df.reset_index(drop=True)

# Validation
assert not inequality_df["Country"].duplicated().any()

for col in inequality_cols:
    assert inequality_df[col].dropna().between(0, 100).all()

print("Cleaned shape:", inequality_df.shape)
print("\nMissing values:")
print(inequality_df.isna().sum())

display(inequality_df.head(10))
display(inequality_df[inequality_df["Country"] == "Australia"])

# Export
inequality_df.to_csv("inequality_clean.csv", index=False)
inequality_oecd.to_csv("inequality_oecd.csv", index=False)

print("Step 9 completed!")


Original shape: (106, 37)
Cleaned shape: (90, 6)

Missing values:
Country                      0
Overall_Digital_Lack_2025    0
Disadvantaged_2025           1
Average_Schools_2025         1
Advantaged_2025              1
Socioeconomic_Gap_2025       1
dtype: int64


,Country,Overall_Digital_Lack_2025,Disadvantaged_2025,Average_Schools_2025,Advantaged_2025,Socioeconomic_Gap_2025
0,Argentina,62.33416,81.31448,69.00422,30.69564,50.61884
1,Armenia,55.76785,56.43072,54.96634,55.63969,0.79103
2,Australia,11.75857,22.83478,12.09888,1.02083,21.81395
3,Austria,19.22027,30.11544,12.24549,22.49068,7.62476
4,Azerbaijan,47.73195,53.87615,44.27895,48.06405,5.81210
5,Belgium,10.77647,17.13164,11.08953,4.35994,12.77170
6,Brazil,35.73083,48.93571,40.89952,11.55333,37.38238
7,Brunei Darussalam,44.16915,56.85573,56.77211,6.24704,50.60869
8,B-S-J-Z (China),32.06114,43.88036,33.56470,17.57124,26.30912
9,Bulgaria,11.66984,6.94800,10.69033,17.75012,-10.80212


,Country,Overall_Digital_Lack_2025,Disadvantaged_2025,Average_Schools_2025,Advantaged_2025,Socioeconomic_Gap_2025
2,Australia,11.75857,22.83478,12.09888,1.02083,21.81395


Step 9 completed!


# Step 10: Clean Resource-Science Relationship

In [10]:
relationship_raw = pd.read_excel(
    "I_B1_4_187_resource_science_relationship.xlsx",
    sheet_name="Sheet1",
    header=None
)

print("Original shape:", relationship_raw.shape)

# Columns from OECD countries section:
# 0 = Resource category
# 1 = Resource indicator
# 2 = Correlation with science performance (r)
# 3 = p-value for r
# 4 = Partial correlation
# 5 = p-value for partial correlation

relationship_df = relationship_raw.iloc[
    5:71, [0, 1, 2, 3, 4, 5]
].copy()

relationship_df.columns = [
    "Resource_Category",
    "Resource_Indicator",
    "Science_Correlation",
    "Science_P_Value",
    "Science_Partial_Correlation",
    "Science_Partial_P_Value"
]

# Forward-fill category headings
relationship_df["Resource_Category"] = (
    relationship_df["Resource_Category"].ffill()
)

# Clean text
for col in ["Resource_Category", "Resource_Indicator"]:
    relationship_df[col] = (
        relationship_df[col]
        .astype("string")
        .str.strip()
    )

numeric_cols = [
    "Science_Correlation",
    "Science_P_Value",
    "Science_Partial_Correlation",
    "Science_Partial_P_Value"
]

for col in numeric_cols:
    relationship_df[col] = pd.to_numeric(
        relationship_df[col],
        errors="coerce"
    )

# Keep valid indicator rows
relationship_df = relationship_df.dropna(
    subset=["Resource_Indicator"]
)

relationship_df = relationship_df.dropna(
    subset=[
        "Science_Correlation",
        "Science_Partial_Correlation"
    ],
    how="all"
)

# Add significance indicators
relationship_df["Significant_R"] = (
    relationship_df["Science_P_Value"] < 0.05
)

relationship_df["Significant_Partial_R"] = (
    relationship_df["Science_Partial_P_Value"] < 0.05
)

relationship_df = relationship_df.reset_index(drop=True)

# Validation
for col in [
    "Science_Correlation",
    "Science_Partial_Correlation"
]:
    assert relationship_df[col].dropna().between(-1, 1).all()

print("Cleaned shape:", relationship_df.shape)
print("\nMissing values:")
print(relationship_df.isna().sum())

display(relationship_df.head(15))

# Export separately: not country-level data
relationship_df.to_csv(
    "resource_science_relationship_clean.csv",
    index=False
)

print("Step 10 completed!")


Original shape: (78, 74)
Cleaned shape: (66, 8)

Missing values:
Resource_Category              0
Resource_Indicator             0
Science_Correlation            0
Science_P_Value                0
Science_Partial_Correlation    0
Science_Partial_P_Value        0
Significant_R                  0
Significant_Partial_R          0
dtype: int64


,Resource_Category,Resource_Indicator,Science_Correlation,Science_P_Value,Science_Partial_Correlation,Science_Partial_P_Value,Significant_R,Significant_Partial_R
0,Shortage of education staff,Index of shortage of education staff,-0.02359,0.88828,0.29378,0.07759,False,False
1,Shortage of education staff,Gap in the index of shortage of education staf...,-0.04870,0.77523,0.08001,0.64274,False,False
2,Shortage of education staff,Instruction hindered to some extent or a lot b...,0.17238,0.30071,0.23103,0.16890,False,False
3,Shortage of education staff,Instruction hindered to some extent or a lot b...,0.19213,0.24785,0.28500,0.08732,False,False
4,Shortage of education staff,Instruction hindered to some extent or a lot b...,-0.08972,0.59514,0.21052,0.21105,False,False
5,Shortage of education staff,Instruction hindered to some extent or a lot b...,-0.16117,0.34609,0.16835,0.31923,False,False
6,Teaching staff,Student-teacher ratio,-0.43890,0.02302,-0.34596,0.06508,True,False
7,Teaching staff,Language-of-instruction class size,-0.37854,0.04060,-0.05040,0.76764,True,False
8,Teaching staff,Class size in science,-0.38226,0.04494,-0.01957,0.91121,True,False
9,Teaching staff,Percentage of fully certified teachers,0.21330,0.22581,0.25978,0.14429,False,False


Step 10 completed!


# Step 11: Merge Science, Digital and Inequality

In [12]:
# Check duplicate country names
for name, df in {
    "Science": country_df,
    "Digital": digital_df,
    "Inequality": inequality_df
}.items():
    duplicates = df["Country"].duplicated().sum()
    print(f"{name} duplicate countries: {duplicates}")
    assert duplicates == 0

# Start from science data
merged_df = country_df.copy()

# Merge digital resource shortage
merged_df = merged_df.merge(
    digital_df,
    on="Country",
    how="left",
    validate="one_to_one"
)

# Merge socioeconomic inequality
merged_df = merged_df.merge(
    inequality_df.drop(
        columns=["Overall_Digital_Lack_2025"]
    ),
    on="Country",
    how="left",
    validate="one_to_one"
)

print("Merged dataset shape:", merged_df.shape)

# Check availability of important variables
important_cols = [
    "Science_Change_2022_2025",
    "Digital_Lack_Change",
    "Digital_Quality_Change",
    "Socioeconomic_Gap_2025"
]

print("\nAvailable observations:")
print(merged_df[important_cols].notna().sum())

# Create separate datasets for analysis

# Change in science vs change in digital shortage
science_digital_df = merged_df.dropna(
    subset=[
        "Science_Change_2022_2025",
        "Digital_Lack_Change"
    ]
).copy()

# Science in 2025 vs socioeconomic resource gap
science_inequality_df = merged_df.dropna(
    subset=[
        "2025",
        "Socioeconomic_Gap_2025"
    ]
).copy()

print("\nScience-Digital sample size:", len(science_digital_df))
print("Science-Inequality sample size:", len(science_inequality_df))

display(merged_df.head())

# Export
merged_df.to_csv("merged_pisa_data.csv", index=False)

science_digital_df.to_csv(
    "science_digital_analysis.csv",
    index=False
)

science_inequality_df.to_csv(
    "science_inequality_analysis.csv",
    index=False
)

print("Step 11 completed!")


Science duplicate countries: 0
Digital duplicate countries: 0
Inequality duplicate countries: 0
Merged dataset shape: (90, 20)

Available observations:
Science_Change_2022_2025    76
Digital_Lack_Change         77
Digital_Quality_Change      77
Socioeconomic_Gap_2025      88
dtype: int64

Science-Digital sample size: 76
Science-Inequality sample size: 88


,Country,2006,2009,2012,2015,2018,2022,2025,Science_Change_2022_2025,Science_Change_2006_2025,Digital_Lack_2022,Digital_Quality_2022,Digital_Lack_2025,Digital_Quality_2025,Digital_Lack_Change,Digital_Quality_Change,Disadvantaged_2025,Average_Schools_2025,Advantaged_2025,Socioeconomic_Gap_2025
0,Argentina,391.0,401.0,406.0,NaN,404.0,406.0,393,-13.0,2.0,67.72479,67.53960,62.33416,60.49732,-5.39063,-7.04228,81.31448,69.00422,30.69564,50.61884
1,Armenia,NaN,NaN,NaN,NaN,NaN,NaN,369,NaN,NaN,NaN,NaN,55.76785,53.62173,NaN,NaN,56.43072,54.96634,55.63969,0.79103
2,Australia,527.0,527.0,521.0,510.0,503.0,507.0,509,2.0,-18.0,9.92705,9.85572,11.75857,11.36550,1.83152,1.50978,22.83478,12.09888,1.02083,21.81395
3,Austria,511.0,NaN,506.0,495.0,490.0,491.0,497,6.0,-14.0,25.90449,20.69598,19.22027,19.59430,-6.68422,-1.10168,30.11544,12.24549,22.49068,7.62476
4,Azerbaijan,382.0,373.0,NaN,NaN,NaN,NaN,409,NaN,27.0,NaN,NaN,47.73195,44.48527,NaN,NaN,53.87615,44.27895,48.06405,5.81210


Step 11 completed!


# Step 12: Final Validation and Summary

In [13]:
print("=" * 55)
print("STAT3011 — DATA CLEANING SUMMARY")
print("=" * 55)

datasets = {
    "Science Wide": country_df,
    "Science Long": science_long,
    "Digital Resource": digital_df,
    "Socioeconomic Inequality": inequality_df,
    "Resource–Science Relationship": relationship_df,
    "Merged Dataset": merged_df,
    "Science-Digital Analysis": science_digital_df,
    "Science-Inequality Analysis": science_inequality_df
}

for name, df in datasets.items():
    print(
        f"{name:<32} "
        f"Rows: {len(df):>4} "
        f"Columns: {len(df.columns):>3}"
    )

print("\n" + "=" * 55)
print("CORRELATION CHECKS")
print("=" * 55)

# Correlation 1: science change vs shortage change
r1 = science_digital_df[
    "Science_Change_2022_2025"
].corr(
    science_digital_df["Digital_Lack_Change"]
)

# Correlation 2: science score vs socioeconomic gap
r2 = science_inequality_df["2025"].corr(
    science_inequality_df["Socioeconomic_Gap_2025"]
)

print(f"Science change vs digital shortage change: {r1:.3f}")
print(f"Science score vs socioeconomic shortage gap: {r2:.3f}")

print("\n" + "=" * 55)
print("DATA QUALITY CHECKS")
print("=" * 55)

assert not merged_df["Country"].duplicated().any()

assert len(science_digital_df) > 2
assert len(science_inequality_df) > 2

# Check that the merge did not multiply science rows
assert len(merged_df) == len(country_df)

# Verify matching country data between tables
digital_check = merged_df[
    ["Country", "Digital_Lack_2025"]
].merge(
    inequality_df[
        ["Country", "Overall_Digital_Lack_2025"]
    ],
    on="Country",
    how="inner"
)

difference = (
    digital_check["Digital_Lack_2025"] -
    digital_check["Overall_Digital_Lack_2025"]
)

assert difference.dropna().abs().max() < 0.01

print("All validation checks passed!")

print("\nMissing values in merged dataset:")
print(merged_df.isna().sum().sort_values(ascending=False))

print("\nData cleaning pipeline completed successfully!")


STAT3011 — DATA CLEANING SUMMARY
Science Wide                     Rows:   90 Columns:  10
Science Long                     Rows:  474 Columns:   3
Digital Resource                 Rows:   90 Columns:   7
Socioeconomic Inequality         Rows:   90 Columns:   6
Resource–Science Relationship    Rows:   66 Columns:   8
Merged Dataset                   Rows:   90 Columns:  20
Science-Digital Analysis         Rows:   76 Columns:  20
Science-Inequality Analysis      Rows:   88 Columns:  20

CORRELATION CHECKS
Science change vs digital shortage change: -0.303
Science score vs socioeconomic shortage gap: -0.381

DATA QUALITY CHECKS
All validation checks passed!

Missing values in merged dataset:
2006                        37
Science_Change_2006_2025    37
2012                        32
2015                        29
2009                        28
2018                        16
Science_Change_2022_2025    14
2022                        14
Digital_Lack_Change         13
Digital_Lack_2022       